# Импорт библиотек

In [ ]:
!pip install wordcloud

In [ ]:
!pip install hdbscan

In [ ]:
import pandas as pd
import numpy as np
import re
import math

import matplotlib.pyplot as plt
import seaborn as sns
import hdbscan
import nltk

from wordcloud import WordCloud
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import Normalizer
from nltk.corpus import stopwords

In [ ]:
sns.set(style="whitegrid")

# EDA

In [ ]:
df = pd.read_csv('data.csv')

In [ ]:
df.head()

- title – заголовок книги
- genre – жанр книги
- summary – описание книги

Сразу удалим колонку index

In [ ]:
df = df.drop(columns=['index'])

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

## Пропуски

In [ ]:
df.isnull().sum()

## Дубликаты

In [ ]:
df.duplicated().sum()

In [ ]:
df[df.duplicated()]

In [ ]:
df = df.drop_duplicates()

In [ ]:
df.describe(include="O")

In [ ]:
genre_counts = df["genre"].value_counts()
genre_percent = df["genre"].value_counts(normalize=True) * 100

In [ ]:
plt.figure()
genre_counts.plot(kind="bar")
plt.title("Распределение книг по жанрам")
plt.xlabel("Жанр")
plt.ylabel("Количество")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Вывод:
- Датасет имеет сильный дисбаланс классов
- При обучении модели классификации жанра возможны: смещение модели в сторону крупных классов; низкий recall для малых жанров
- Для корректного моделирования могут потребоваться: стратифицированное разбиение, class weights, oversampling / undersampling, использование F1-macro вместо accuracy.

In [ ]:
df["summary"] = df["summary"].astype(str)

In [ ]:
df["char_length"] = df["summary"].apply(len)
df["word_length"] = df["summary"].apply(lambda x: len(x.split()))

In [ ]:
df["char_length"].describe()

Вывод:
- Распределение сильно асимметрично
- Большинство описаний имеют умеренную длину
- Присутствует значительное число длинных текстов
- Данные не распределены нормально

## Артефакты

In [ ]:
df[df.char_length == 11]

Видно, что в summary есть артефакт ==Receptio, посмотрим на все

In [ ]:
mask_wiki = df['summary'].str.startswith(' ==', na=False)

In [ ]:
wiki_rows = df[mask_wiki]

In [ ]:
wiki_rows

In [ ]:
print("Количество строк, начинающихся с '==':", wiki_rows.shape[0])
print("Доля от датасета:", wiki_rows.shape[0] / len(df))

Найдем в GoodReads (откуда брал автор данные) summary для данных книг

In [ ]:
df.loc[
    df.title == "The Caverns of Kalte", "summary"] = "You are Lone Wolf - last of the Kai Lords. Shocking news has just reached your homeland that Vonotar the Traitor still lives and now rules over the Ice Barbarians of Kalte. The King has vowed to your people that Vonotar will be brought to justice for his crimes. But it is a promise that only you can fulfil. In THE CAVERNS OF KALTE, you must brave the terrible dangers of the ice kingdom in your quest to capture your most hated foe. But be warned! It is a challenge that will test your skill and endurance to the very limit. The LONE WOLF adventures are a unique interactive fantasy series - each episode can be played separately or you can combine them all to create a fantastic role-playing epic."

In [ ]:
df.loc[
    df.title == "Guardians of Ga'Hoole Book 4: The Siege", "summary"] = "Soren's beloved mentor, Ezylryb, is finally back at the Great Ga'Hoole Tree. But all is not well. There's a war between good and evil in the owl kingdom. On one side is a group led by Soren's fearsome brother, Kludd, who wears a terrifying metal mask to cover his battle-scarred face. On the other side are the owls of the Great Ga'Hoole Tree, who must fight to protect their legendary tree from Kludd's attacks. Soren, his friends, and the other owls at the Great Ga'Hoole Tree enter into fierce combat against Kludd's forces. They win a major battle, but warfare is not over yet."

In [ ]:
df.loc[
    df.title == "Jacob Have I Loved", "summary"] = "Soren's beloved mentor, Ezylryb, is finally back at the Great Ga'Hoole Tree. But all is not well. There's a war between good and evil in the owl kingdom. On one side is a group led by Soren's fearsome brother, Kludd, who wears a terrifying metal mask to cover his battle-scarred face. On the other side are the owls of the Great Ga'Hoole Tree, who must fight to protect their legendary tree from Kludd's attacks. Soren, his friends, and the other owls at the Great Ga'Hoole Tree enter into fierce combat against Kludd's forces. They win a major battle, but warfare is not over yet."

In [ ]:
df.loc[
    df.title == "The Eyes of Darkness", "summary"] = "Esau have I hated . . . Sara Louise Bradshaw is sick and tired of her beautiful twin Caroline. Ever since they were born, Caroline has been the pretty one, the talented one, the better sister. Even now, Caroline seems to take everything: Louise's friends, their parents' love, her dreams for the future. For once in her life, Louise wants to be the special one. But in order to do that, she must first figure out who she is . . . and find a way to make a place for herself outside her sister's shadow."

In [ ]:
df["char_length"] = df["summary"].apply(len)
df["word_length"] = df["summary"].apply(lambda x: len(x.split()))

In [ ]:
df.char_length.describe()

In [ ]:
df[df.char_length == 17]

In [ ]:
df.loc[
    df.title == "Deathstalker Destiny", "summary"] = "Owen Deathstalker’s greatest love—Hazel d’Ark—has been abducted by the Blood Runners, a culture dedicated to the extremes of genetic experimentation. Stranded in a mission on Lachrymae Christi, Owen busies himself with the task of ensuring the survival of the leper colony living there, awaiting an opportunity to rescue Hazel…or avenge her death."

In [ ]:
df["char_length"] = df["summary"].apply(len)
df["word_length"] = df["summary"].apply(lambda x: len(x.split()))

In [ ]:
df.word_length.describe()

In [ ]:
df[df["summary"].str.contains("Plot outline description", na=False)]

Еще есть артефакты, заменим их

In [ ]:
df.loc[
    df.title == "Missing Men of Saturn", "summary"] = "\"We Go Anywhere\" was the legend scrawled on the battered hull of the ALBATROSS, one of the worst old tubs in space. To Dale Sutton, the biggest man on campus at the Space Academy, it was a slap in the face to be ordered to such a crate. But his biggest shock came when orders set the ALBATROSS and its two companion ships on a course that let straight to the dreaded planet Saturn. No one had ever come back from Saturn, yet everyone knew the story of Captain Dearborn who had led the first and only expedition to the ringed planet a century earlier. His diary was the record of a steadily losing battle against the unknown as one by one, the little party had vanished. Now, a hundred years later, the superstitious crew of the ALBATROSS found it impossible to rid themselves of the feeling that the same catastrophe that had wiped out the previous expedition would strike again. They had hardly been settled a day in Dearborn's old underground quarters on Titan, Saturn's largest satellite, when their gnawing fears began to materialize. First, the loss of all their guns when the lights suddenly and inexplicably faded, then the disappearance of the first man! But greater and more deadly horrors were yet to panicky moments of groping though ghastly underground caves, the appearance of a face bearing the same twisted features of the illustrious Captain Dearborn, a collision that sends Titan up in a blaze of destruction, and the final landing on Saturn, a planet heaving with volcanos and covered with streams of molten lava. Philip Latham's portrayal of life on a planet about whose conditions few have ventured a guess is a tale guaranteed to make the reader as numb with terror as the men the author writes about. Philip Latham was a pen name used by Dr. Robert S. Richardson (1902 – 1981). He could support the suppositions that are the basis of his science fiction novels with accepted scientific theories. For he was an author who was in the business of “watching the stars.” An astronomer at Mount Wilson and Palomar Observatories beginning in 1931, he started writing for magazines in the early forties. His work won such wide respect that he also had a college textbook on astronomy to his credit. Movie producers as well as publishers found Dr. Richardson’s experience too good to pass up. He gave technical assistance to a number of studios on pictures such as Destination Moon , and he wrote an article describing his work on the science fiction thriller When Worlds Collide."

In [ ]:
df.loc[
    df.title == "The Unwilling Warlord", "summary"] = "A STAR RISES IN THE SOUTH. When the foreigners confronted Sterren in Ethshar of the Spices he was uneasy; when they all but abducted him, taking him to an obscure kingdom in the south, he knew he was in a terrible predicament."

In [ ]:
df.loc[
    df.title == "Stadium Beyond the Stars", "summary"] = "n route to the Center of the Galaxy for the Interstellar Olympic Games, the HELLAS, carrying Earth's team, intercepts a mysterious space ship, apparently derelict. Steve Frazer, champion spacesuit racer, volunteers to investigate. Once aboard, he discovers astonishing evidence of an intelligent nonhuman race that can speak by telepathy and disappear at will - a race superior in some ways to human beings. Stunned, Steve returns to the HELLAS to find that no one believes his startling story. His attempts to prove that he is telling the truth plunge Steve quickly into the midst of interstellar conflict and intrigue. Disqualified from the Games on a trumped-up charge, Steve soon realizes that someone very powerful thinks he knows too much. Tightly written and intensely dramatic, the story sweeps to the outermost reaches of the galaxy. Its picture of the Games with their brilliant color and keen competition is entirely new to the pages of science fiction. Milton Lesser was raised in Brooklyn and attended the College of William and Mary. After several years writing science fiction under his given name, including four books for the Winston Science Fiction series, he legally adopted the pen name Stephen Marlowe. He authored more than fifty novels, including nearly two dozen featuring globe-trotting private eye Chester Drum."

In [ ]:
df.loc[
    df.title == "Grania: She-King of the Irish Seas", "summary"] = "Here is an extraordinary novel about real-life Irish chieftain Grace O Malley. From Morgan Llywelyn, bestselling author of Lion of Ireland and the Irish Century novels, comes the story of a magnificent, sixteenth-century heroine whose spirit and passion are the spirit and passion of Ireland itself. Grania (Gaelic for Grace) is no ordinary female. And she lives in extraordinary times. For even as Grania rises as her clan's unofficial head and breadwinner and learns to love a man, she enters a lifelong struggle against the English forces of Queen Elizabeth -- her nemesis and alter ego. Elizabeth intends to destroy Grania's piracy and shipping empire--and so subjugate Ireland once and for all. But Grania, aided by Tigernan, her faithful (and secretly adoring) lieutenant, has no choice but to fight back. The story of her life is the story of Ireland's fight for solidarity and survival--but it's also the story of Grania's growing ability to love and be strong at the same time. Morgan Llywelyn has written a rich, historically accurate, and passionate novel of divided Ireland -- and of one brave woman who is Ireland herself."

Последнее удалим так как название и описание книги было неправильно спаршено

In [ ]:
df.drop(2485, inplace=True)

In [ ]:
df = df.reset_index(drop=True)

In [ ]:
df["char_length"] = df["summary"].apply(len)
df["word_length"] = df["summary"].apply(lambda x: len(x.split()))

In [ ]:
df.word_length.describe()

## Распределения

Распределение длины summary

In [ ]:
df["summary_length"] = df["summary"].str.len()

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df["summary_length"], bins=50, kde=True)
plt.title("Распределение длины описаний")
plt.xlabel("Длина текста")
plt.ylabel("Количество")
plt.show()

Вывод:
- Сильная правосторонняя асимметрия
- Пик плотности на 500 - 1500
- Длинный хвост

Boxplot для поиска выбросов

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(x=df["summary_length"])
plt.title("Выбросы длины summary")
plt.show()

Вывод:
- Медиана находится примерно около 1500
- Основная часть данных (IQR) (300 - 2500)
- Большое количество выбросов, которые нужно будет удалять или ограничивать

Средняя длина текста по жанрам

In [ ]:
genre_summary = df.groupby("genre")["summary_length"].mean().sort_values()

In [ ]:
plt.figure(figsize=(10, 6))
genre_summary.plot(kind="bar")
plt.title("Средняя длина описания по жанрам")
plt.xlabel("Жанр")
plt.ylabel("Средняя длина")
plt.show()

Вывод:
- Жанр влияет на длину текста

Корреляция числовых признаков

In [ ]:
df["title_length"] = df["title"].str.len()

In [ ]:
corr = df[["summary_length", "title_length"]].corr()

In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, cmap="coolwarm")
plt.title("Корреляция признаков")
plt.show()

Признаки некоррелируют

# Предобработка данных

## title

In [ ]:
nltk.download('stopwords')

In [ ]:
stop_words = stopwords.words('english')

Очистим текст

In [ ]:
def preprocess_text(text):
    text = text.lower()

    text = re.sub(r'[^a-zA-Z\s]', '', text)

    words = text.split()

    words = [word for word in words if word not in stop_words]

    text = " ".join(words)

    return text

In [ ]:
df['clean_title'] = df['title'].apply(preprocess_text)

df[['title', 'clean_title']].head()

## Подготовка Bag-of-Words (CountVectorizer)

In [ ]:
count_vectorizer = CountVectorizer()
X_count = count_vectorizer.fit_transform(df['clean_title'])

In [ ]:
X_count.shape

In [ ]:
feature_names_count = count_vectorizer.get_feature_names_out()

In [ ]:
feature_names_count[:20]

In [ ]:
count_df = pd.DataFrame(
    X_count.toarray(),
    columns=feature_names_count
)
count_df.head()

Подготовка TF-IDF

In [ ]:
tfidf_vectorizer = TfidfVectorizer()
X_tfidf = tfidf_vectorizer.fit_transform(df['clean_title'])

In [ ]:
X_tfidf.shape

In [ ]:
feature_names_tfidf = tfidf_vectorizer.get_feature_names_out()

In [ ]:
feature_names_tfidf[:20]

In [ ]:
tfidf_df = pd.DataFrame(
    X_tfidf.toarray(),
    columns=feature_names_tfidf
)

tfidf_df.head()

## summary

In [ ]:
df["summary_clean"] = df["summary"].apply(preprocess_text)

In [ ]:
min_chars = 50
mask_len = df["summary_clean"].str.len() >= min_chars

In [ ]:
df_work = df.loc[mask_len].copy()
print("Размер после фильтра коротких summary:", df_work.shape)

# Кластеризация по title

## Кластеризация CountVectorizer

In [ ]:
kmeans_count = KMeans(n_clusters=10, random_state=42)

In [ ]:
clusters_count = kmeans_count.fit_predict(X_count)

In [ ]:
df['cluster_count'] = clusters_count

## Кластеризация TF-IDF

In [ ]:
kmeans_tfidf = KMeans(n_clusters=10, random_state=42)

In [ ]:
clusters_tfidf = kmeans_tfidf.fit_predict(X_tfidf)

In [ ]:
df['cluster_tfidf'] = clusters_tfidf

## Визуализация кластеров

### Для CountVectorizer

In [ ]:
pca = PCA(n_components=2)

X_count_pca = pca.fit_transform(X_count.toarray())

plt.figure(figsize=(8, 6))
plt.scatter(X_count_pca[:, 0], X_count_pca[:, 1], c=df['cluster_count'], cmap='tab10')

plt.title("KMeans кластеризация (CountVectorizer)")
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")

plt.show()

In [ ]:
sil_count = silhouette_score(X_count, clusters_count)
sil_count

Качество разбиения очень низкое

## Для TF-IDF

In [ ]:
X_tfidf_pca = pca.fit_transform(X_tfidf.toarray())

plt.figure(figsize=(8, 6))
plt.scatter(X_tfidf_pca[:, 0], X_tfidf_pca[:, 1], c=df['cluster_tfidf'], cmap='tab10')

plt.title("KMeans кластеризация (TF-IDF)")
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")

plt.show()

In [ ]:
sil_tfidf = silhouette_score(X_tfidf, clusters_tfidf)
sil_tfidf

Качество разбиения низкое

## Пересечение с жанрами

In [ ]:
ari_count = adjusted_rand_score(df['genre'], df['cluster_count'])
ari_tfidf = adjusted_rand_score(df['genre'], df['cluster_tfidf'])

In [ ]:
nmi_count = normalized_mutual_info_score(df['genre'], df['cluster_count'])
nmi_tfidf = normalized_mutual_info_score(df['genre'], df['cluster_tfidf'])

In [ ]:
print("ARI Count:", ari_count)
print("ARI TF-IDF:", ari_tfidf)
print("NMI Count:", nmi_count)
print("NMI TF-IDF:", nmi_tfidf)

Кластеризация практически случайная, нет взаимосвязи между жанрами

In [ ]:
pd.crosstab(df['genre'], df['cluster_tfidf'])

Почти все объекты в одном кластере (7)

Вывод:
- тексты очень похожи по словарю
- признаки TF-IDF не разделяют жанры

Кластеризация может объяснять кластеры иначе:
- тематике
- стилю названий
- ключевым словам

## TF-IDF c N-граммами

In [ ]:
vectorizer = TfidfVectorizer(
    stop_words='english',
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.7
)

X = vectorizer.fit_transform(df['clean_title'])

In [ ]:
X.shape

In [ ]:
scores = []

for k in range(5, 31):
    model = KMeans(n_clusters=k, random_state=42)

    labels = model.fit_predict(X)

    score = silhouette_score(X, labels)

    scores.append(score)

In [ ]:
plt.plot(range(5, 31), scores)

plt.xlabel("Количество кластеров")
plt.ylabel("Silhouette score")

plt.title("Подбор числа кластеров")

plt.show()

In [ ]:
kmeans = KMeans(n_clusters=10, random_state=42)

In [ ]:
clusters = kmeans.fit_predict(X)

In [ ]:
df["cluster"] = clusters

In [ ]:
pca = PCA(n_components=2)

X_reduced = pca.fit_transform(X.toarray())

plt.figure(figsize=(8, 6))

plt.scatter(
    X_reduced[:, 0],
    X_reduced[:, 1],
    c=df['cluster'],
    cmap='tab20'
)

plt.title("Кластеры заголовков (TF-IDF + N-grams)")

plt.show()

In [ ]:
num_clusters = df["cluster"].nunique()

plt.figure(figsize=(16, 12))

for i in range(num_clusters):
    text = " ".join(df[df["cluster"] == i]["clean_title"])

    wordcloud = WordCloud(
        width=800,
        height=400,
        background_color="white",
        colormap="viridis"
    ).generate(text)

    plt.subplot(4, 3, i + 1)
    plt.imshow(wordcloud)
    plt.axis("off")
    plt.title(f"Cluster {i}")

plt.tight_layout()
plt.show()

Вывод:
- Кластеры формируются по темам и образам, а не по жанра
- Несколько кластеров относятся к фэнтези и мистике, что объясняет их смешивание
- Большинство кластеров пересекаются по тематике

Кластер 0 – книги с фэнтезийной и приключенческой тематикой

Кластер 1 – эпическое фэнтези

Кластер 2 – исторические истории, путешествия, мистические события

Кластер 3 – детективы

Кластер 4 – приключения, экспедиции, исследование неизвестных территорий

Кластер 5 – человеческие отношения, биографии, жизненные истории

Кластер 6 – жанр horror

Кластер 7 – сверхъестественные существа, мистические силы, фэнтези

Кластер 8 – мистические события, тёмная фантастика, триллеры

Кластер 9 – фэнтези-кластер

In [ ]:
genres = df["genre"].unique()

plt.figure(figsize=(16, 12))

for i, g in enumerate(genres):
    text = " ".join(df[df["genre"] == g]["clean_title"])

    wordcloud = WordCloud(
        width=800,
        height=400,
        background_color="white",
        colormap="plasma"
    ).generate(text)

    plt.subplot(4, 3, i + 1)
    plt.imshow(wordcloud)
    plt.axis("off")
    plt.title(g)

plt.tight_layout()
plt.show()

Жанры действительно имеют характерные тематические слова, однако между ними существует частичное пересечение лексики

Некоторые слова встречаются во многих жанрах:
- world
- story
- night
- dark
- life

In [ ]:
ari = adjusted_rand_score(df['genre'], df['cluster'])
nmi = normalized_mutual_info_score(df['genre'], df['cluster'])

print("ARI:", ari)
print("NMI:", nmi)

In [ ]:
pd.crosstab(df["genre"], df["cluster"])

## Улучшенный вариант

In [ ]:
vectorizer = TfidfVectorizer(
    stop_words='english',
    max_df=0.7,
    min_df=3,
    ngram_range=(1, 2)
)

In [ ]:
X = vectorizer.fit_transform(df['clean_title'])

### SVD

In [ ]:
svd = TruncatedSVD(
    n_components=100,
    random_state=42
)

In [ ]:
X_svd = svd.fit_transform(X)

In [ ]:
svd.explained_variance_ratio_.sum()

### Kmeans

Так как были замечены частые слова, уберем их перед кластеризацией и посмотрим что получится

In [ ]:
kmeans = KMeans(n_clusters=10, random_state=42)

In [ ]:
clusters = kmeans.fit_predict(X_svd)

In [ ]:
df["cluster"] = clusters

In [ ]:
num_clusters = df["cluster"].nunique()

plt.figure(figsize=(16, 12))

for i in range(num_clusters):
    text = " ".join(df[df["cluster"] == i]["clean_title"])

    wordcloud = WordCloud(
        width=800,
        height=400,
        background_color="white",
        colormap="viridis"
    ).generate(text)

    plt.subplot(4, 3, i + 1)
    plt.imshow(wordcloud)
    plt.axis("off")
    plt.title(f"Cluster {i}")

plt.tight_layout()
plt.show()

Вывод:
- Кластер 0 объединяет книги с героями и драматическими сюжетами
- Кластер 1 характерен для фэнтези и мистики
- Кластер 2 реалистичный кластер
- Кластер 3 типичная фэнтезийная лексика
- Кластер 4 детективный жанр
- Кластер 5 мистические истории и триллеры
- Кластер 6 high fantasy
- Кластер 7 военные истории, эпические сюжеты
- Кластер 8 научная фантастика
- Кластер 9 dark fantasy

In [ ]:
pd.crosstab(df["cluster"], df["genre"])

### Agglomerative clustering

In [ ]:
agg = AgglomerativeClustering(
    n_clusters=10,
    linkage="ward"
)

In [ ]:
labels_agg = agg.fit_predict(X_svd)

In [ ]:
print("ARI:", adjusted_rand_score(df["genre"], labels_agg))
print("NMI:", normalized_mutual_info_score(df["genre"], labels_agg))
print("Silhouette:", silhouette_score(X_svd, labels_agg))

In [ ]:
pd.crosstab(labels_agg, df["genre"])

In [ ]:
texts = df["title"]

In [ ]:
clusters = sorted(set(labels_agg))

In [ ]:
plt.figure(figsize=(16, 10))

for i, cluster in enumerate(clusters):
    cluster_text = " ".join(texts[labels_agg == cluster])

    wc = WordCloud(
        width=800,
        height=400,
        background_color="black",
        colormap="viridis"
    ).generate(cluster_text)

    plt.subplot(3, 4, i + 1)
    plt.imshow(wc)
    plt.axis("off")
    plt.title(f"Cluster {cluster}")

plt.tight_layout()
plt.show()

Вывод:
- Кластер 0 эпические приключения
- Кластер 1 исторические хроники
- Кластер 2 бытовые истории
- Кластер 3 мистические сюжеты
- Кластер 4 научная фантастика
- Кластер 5 dark fantasy
- Кластер 6 постапокалиптические истории
- Кластер 7 dark fantasy / horror
- Кластер 8 medieval fantasy
- Кластер 9 horror

### HDBSCAN

In [ ]:
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=20,
    metric='euclidean'
)

In [ ]:
labels_hdb = clusterer.fit_predict(X_svd)

In [ ]:
print("Количество кластеров:", len(set(labels_hdb)) - (1 if -1 in labels_hdb else 0))

In [ ]:
mask = labels_hdb != -1

In [ ]:
print("ARI:", adjusted_rand_score(df["genre"][mask], labels_hdb[mask]))
print("NMI:", normalized_mutual_info_score(df["genre"][mask], labels_hdb[mask]))

In [ ]:
texts = df["title"]

In [ ]:
clusters = sorted(set(labels_hdb))

In [ ]:
clusters = [c for c in clusters if c != -1]

In [ ]:
n_clusters = len(clusters)
cols = 4
rows = math.ceil(n_clusters / cols)

In [ ]:
plt.figure(figsize=(16, rows * 4))

for i, cluster in enumerate(clusters):
    cluster_text = " ".join(texts[labels_hdb == cluster])

    wc = WordCloud(
        width=800,
        height=400,
        background_color="black",
        colormap="viridis"
    ).generate(cluster_text)

    plt.subplot(rows, cols, i + 1)
    plt.imshow(wc)
    plt.axis("off")
    plt.title(f"Cluster {cluster}")

plt.tight_layout()
plt.show()

Вывод:
- Кластер 0 фэнтези-тематика
- Кластер 1 постапокалиптические / военные истории
- Кластер 2 dark fantasy / horror
- Кластер 3 триллер / криминальная проза
- Кластер 4 horror
- Кластер 5 мистические / готические истории
- Кластер 6 современные истории / драма
- Кластер 7 исторические или биографические произведения
- Кластер 8 science fiction
- Кластер 9 историческая и научно-популярная литература

Kmeans – равномерные характер кластеров
Agglomerative – тематический характер кластеров
HDBSCAN – характер кластера наиболее естественный

В# Кластеризация по summary

In [ ]:
DOMAIN_STOPWORDS = {
    "book", "books", "novel", "story", "stories", "read", "reader", "readers", "author", "edition", "isbn", "cover"
}

In [ ]:
vectorizer = TfidfVectorizer(
    max_df=0.7,
    min_df=5,
    stop_words=list(DOMAIN_STOPWORDS) + ["english"],
    ngram_range=(1, 2),
    sublinear_tf=True,
    norm="l2"
)

In [ ]:
X = vectorizer.fit_transform(df_work["summary_clean"])

In [ ]:
n_components = 200
svd = TruncatedSVD(n_components=n_components, random_state=42)

In [ ]:
lsa = make_pipeline(svd, Normalizer(copy=False))
X_svd = lsa.fit_transform(X)

In [ ]:
print("SVD shape:", X_svd.shape)
print("Объясненная дисперсия:", float(svd.explained_variance_ratio_.sum()))

In [ ]:
y_true = df_work["genre"]

In [ ]:
def clustering_report(name: str, labels: np.ndarray, X_emb: np.ndarray, y=None):
    labels = np.asarray(labels)
    print("\n" + "=" * 70)
    print(f"{name}")
    print("=" * 70)

    uniq = np.unique(labels)
    if -1 in uniq:
        mask = labels != -1
        labels_s = labels[mask]
        X_s = X_emb[mask]
    else:
        mask = np.ones(len(labels), dtype=bool)
        labels_s = labels
        X_s = X_emb

    if len(np.unique(labels_s)) >= 2 and len(labels_s) >= 10:
        try:
            sil = silhouette_score(X_s, labels_s)
            print("Silhouette:", float(sil))
        except Exception as e:
            print("Silhouette: не удалось посчитать:", repr(e))
    else:
        print("Silhouette: не считается (слишком мало кластеров/точек после фильтра).")

    if y is not None:
        try:
            ari = adjusted_rand_score(y[mask], labels[mask])
            nmi = normalized_mutual_info_score(y[mask], labels[mask])
            print("ARI:", float(ari))
            print("NMI:", float(nmi))
        except Exception as e:
            print("ARI/NMI: не удалось посчитать:", repr(e))

        try:
            tab = pd.crosstab(labels, y)
            print("\nCrosstab (cluster x genre):")
            print(tab)
        except Exception as e:
            print("Crosstab: не удалось построить:", repr(e))

In [ ]:
def top_terms_per_cluster_kmeans(km_model: KMeans, vectorizer: TfidfVectorizer, top_n=15):
    print("\nТоп-термины по KMeans центроидам (если KMeans обучали на TF-IDF):")
    terms = vectorizer.get_feature_names_out()
    centers = km_model.cluster_centers_
    for i in range(centers.shape[0]):
        top = np.argsort(centers[i])[-top_n:][::-1]
        print(i, ":", ", ".join(terms[top]))

In [ ]:
def top_terms_by_mean_tfidf(labels: np.ndarray, X_tfidf, vectorizer: TfidfVectorizer, top_n=15, ignore_noise=True):
    labels = np.asarray(labels)
    terms = vectorizer.get_feature_names_out()
    clusters = sorted(set(labels))
    if ignore_noise and (-1 in clusters):
        clusters = [c for c in clusters if c != -1]

    out = {}
    for c in clusters:
        idx = np.where(labels == c)[0]
        if len(idx) == 0:
            continue
        mean_vec = X_tfidf[idx].mean(axis=0)
        mean_vec = np.asarray(mean_vec).ravel()
        top_idx = mean_vec.argsort()[-top_n:][::-1]
        out[c] = [(terms[i], float(mean_vec[i])) for i in top_idx]
    return out


In [ ]:
def plot_wordclouds_from_clusters(labels: np.ndarray, texts: pd.Series, title_prefix="Cluster", max_words=120):
    labels = np.asarray(labels)
    clusters = sorted(set(labels))
    clusters = [c for c in clusters if c != -1]

    cols = 4
    rows = math.ceil(len(clusters) / cols)
    plt.figure(figsize=(16, rows * 4))

    for i, c in enumerate(clusters):
        cluster_text = " ".join(texts[labels == c].astype(str).tolist())
        wc = WordCloud(
            width=800,
            height=400,
            background_color="black",
            max_words=max_words
        ).generate(cluster_text)

        plt.subplot(rows, cols, i + 1)
        plt.imshow(wc)
        plt.axis("off")
        plt.title(f"{title_prefix} {c}")

    plt.tight_layout()
    plt.show()

## Kmeans

In [ ]:
kmeans = KMeans(n_clusters=10, random_state=42, n_init=10)

In [ ]:
labels_km = kmeans.fit_predict(X_svd)

In [ ]:
clustering_report("KMeans (TF-IDF -> SVD -> KMeans)", labels_km, X_svd, y_true)

In [ ]:
top_km = top_terms_by_mean_tfidf(labels_km, X, vectorizer, top_n=15)
print("Топ-слова по кластерам (KMeans), средний TF-IDF:")
for c, words in top_km.items():
    print(c, ":", ", ".join([w for w, _ in words]))

In [ ]:
plot_wordclouds_from_clusters(labels_km, df_work["summary_clean"], title_prefix="KMeans", max_words=120)

Вывод:
- Кластер 0 фантастика
- Кластер 1 маркетинговый бейдж
- Кластер 2 научпоп/история/путешествия
- Кластер 4 очень широкий кластер, общая фантастика
- Кластер 5 научная фантастика
- Кластер 6 фандомные тексты / конкретная вселенная
- Кластер 7 разговорные/краткие описания
- Кластер 8 детективный жанр
- Кластер 9 войны, политика, история

Кластеры отражают темы и стиль аннотации, а не строго жанр

## Agglomerative

In [ ]:
agg = AgglomerativeClustering(n_clusters=10, linkage="ward")
labels_agg = agg.fit_predict(X_svd)

In [ ]:
clustering_report("Agglomerative (TF-IDF -> SVD -> Agglomerative)", labels_agg, X_svd, y_true)

In [ ]:
top_agg = top_terms_by_mean_tfidf(labels_agg, X, vectorizer, top_n=15)
print("Топ-слова по кластерам (Agglomerative), средний TF-IDF:")
for c, words in top_agg.items():
    print(c, ":", ", ".join([w for w, _ in words]))

In [ ]:
plot_wordclouds_from_clusters(labels_agg, df_work["summary_clean"], title_prefix="Agglo", max_words=120)

Вывод:
- Кластер 0 “маркетинговые ярлыки”
- Кластер 1 огромный смешанный кластер
- Кластер 2 исторические романы про Англию/Францию
- Кластер 3 научпоп / история / “non-fiction”
- Кластер 4 общий кластер
- Кластер 5 детектив/криминал
- Кластер 6 sci-fi / космос
- Кластер 7 Buffy/Angel (фандом)
- Кластер 8 политика/шпионаж/война
- Кластер 9 современная история

## HDBSCAN

In [ ]:
hdb = hdbscan.HDBSCAN(
    min_cluster_size=30,
    min_samples=10,
    cluster_selection_method="eom"
)

In [ ]:
labels_hdb = hdb.fit_predict(X_svd)

In [ ]:
n_clusters = len(set(labels_hdb)) - (1 if -1 in labels_hdb else 0)
n_noise = int((labels_hdb == -1).sum())
print("HDBSCAN clusters:", n_clusters, "| noise:", n_noise)

In [ ]:
clustering_report("HDBSCAN (TF-IDF -> SVD -> HDBSCAN)", labels_hdb, X_svd, y_true)

In [ ]:
top_hdb = top_terms_by_mean_tfidf(labels_hdb, X, vectorizer, top_n=15, ignore_noise=True)
print("\nТоп-слова по кластерам (HDBSCAN), средний TF-IDF (без noise):")
for c, words in top_hdb.items():
    print(c, ":", ", ".join([w for w, _ in words]))

In [ ]:
plot_wordclouds_from_clusters(labels_hdb, df_work["summary_clean"], title_prefix="HDBSCAN", max_words=120)

Вывод:
- Почти все стало шумом
- Кластер 0 маркетинговые ярлыки и персонажи
- Кластер 1 общий “нарративный шаблон”
- Кластер 2 спорт и немного fantasy/romance

Качество кластеризации сильно зависит от объёма и информативности текста. Кластеризация коротких текстов (title) приводит к слабому разделению данных и образованию крупных смешанных кластеров, поскольку названия книг содержат ограниченный словарь и мало контекста. Кластеризация аннотаций (summary) даёт более качественные результаты, так как длинные тексты содержат больше тематической информации и позволяют алгоритмам выявлять группы произведений по содержанию. Таким образом, для задач тематической кластеризации текстов более информативными являются развернутые описания, тогда как короткие заголовки требуют дополнительной обработки или более сложных методов анализа текста